In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import gc
import torch
from utils import *
from experiment import Experiment
from reasoning_graph import ReasoningGraph
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
# model_name = "Qwen/Qwen3-4B-Thinking-2507"
# model_name = "Qwen/Qwen3-1.7B"
model_name = "Qwen/Qwen2.5-Math-1.5B-Instruct"
num_problems = 1
num_rollouts = 1
temperature = 1.0

In [11]:
dataset_name = 'HuggingFaceH4/MATH-500'
problems = load_problems_from_dataset(dataset_name=dataset_name, num_problems=num_problems)

for i, p in enumerate(problems[:3]):
    print(f"\nProblem {i+1}: {p['question'][:100]}...")
    print(f"Ground truth: {p['ground_truth']}")

Loading 1 problems from HuggingFaceH4/MATH-500:test
{'problem': 'Let $S$ be the set of points $(a,b)$ with $0 \\le a,$ $b \\le 1$ such that the equation\n\\[x^4 + ax^3 - bx^2 + ax + 1 = 0\\]has at least one real root.  Determine the area of the graph of $S.$', 'solution': 'Note that $x = 0$ cannot be a solution of the equation.  Dividing both sides by $x^2,$ we get\n\\[x^2 + ax - b + \\frac{a}{x} + \\frac{1}{x^2} = 0.\\]Let $y = x + \\frac{1}{x}.$  Then $x^2 - yx + 1 = 0.$  The discriminant of this quadratic is\n\\[y^2 - 4,\\]so there is a real root in $x$ as long as $|y| \\ge 2.$\n\nAlso, $y^2 = x^2 + 2 + \\frac{1}{x^2},$ so\n\\[y^2 + ay - (b + 2) = 0.\\]By the quadratic formula, the roots are\n\\[y = \\frac{-a \\pm \\sqrt{a^2 + 4(b + 2)}}{2}.\\]First, we notice that the discriminant $a^2 + 4(b + 2)$ is always positive.  Furthermore, there is a value $y$ such that $|y| \\ge 2$ as long as\n\\[\\frac{a + \\sqrt{a^2 + 4(b + 2)}}{2} \\ge 2.\\]Then $a + \\sqrt{a^2 + 4(b + 2)} \\ge 4,$ or $

In [12]:
problems[0]

{'index': 494,
 'question': 'Let $S$ be the set of points $(a,b)$ with $0 \\le a,$ $b \\le 1$ such that the equation\n\\[x^4 + ax^3 - bx^2 + ax + 1 = 0\\]has at least one real root.  Determine the area of the graph of $S.$',
 'full_answer': 'Note that $x = 0$ cannot be a solution of the equation.  Dividing both sides by $x^2,$ we get\n\\[x^2 + ax - b + \\frac{a}{x} + \\frac{1}{x^2} = 0.\\]Let $y = x + \\frac{1}{x}.$  Then $x^2 - yx + 1 = 0.$  The discriminant of this quadratic is\n\\[y^2 - 4,\\]so there is a real root in $x$ as long as $|y| \\ge 2.$\n\nAlso, $y^2 = x^2 + 2 + \\frac{1}{x^2},$ so\n\\[y^2 + ay - (b + 2) = 0.\\]By the quadratic formula, the roots are\n\\[y = \\frac{-a \\pm \\sqrt{a^2 + 4(b + 2)}}{2}.\\]First, we notice that the discriminant $a^2 + 4(b + 2)$ is always positive.  Furthermore, there is a value $y$ such that $|y| \\ge 2$ as long as\n\\[\\frac{a + \\sqrt{a^2 + 4(b + 2)}}{2} \\ge 2.\\]Then $a + \\sqrt{a^2 + 4(b + 2)} \\ge 4,$ or $\\sqrt{a^2 + 4(b + 2)} \\ge 4 - 

In [13]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)

In [14]:
# Create an experiment
experiment = Experiment()

# Setup new experiment
experiment.setup_new(
    model_name=model_name,
    dataset_name=dataset_name,
    problems=problems,
    num_rollouts=num_rollouts,
    temperature=temperature
)

Setup new experiment at: experiments/experiment_20251109_161042 with config:
{'model_name': 'Qwen/Qwen2.5-Math-1.5B-Instruct', 'dataset_name': 'HuggingFaceH4/MATH-500', 'num_problems': 1, 'num_rollouts': 1, 'temperature': 1.0, 'timestamp': '20251109_161042'}


In [15]:
# Conduct the experiment with initiailized model and tokenizer
experiment.conduct_experiment(model, tokenizer)


Problem 1/1


Problem 494:   0%|          | 0/1 [01:06<?, ?it/s]


TypeError: unsupported operand type(s) for -: 'float' and 'str'

In [ ]:
# Unload model and tokenizer from memory
del model
del tokenizer
gc.collect()

# Release GPU cache (if any)
if torch.cuda.is_available():
    torch.cuda.empty_cache()

: 

In [24]:
tokens = [tokenizer.decode(id) for id in output_ids[len(model_inputs.input_ids[0]):]]
entropies = reasoning_graph.metrics
visualize_tokens(tokens, entropies)

In [ ]:
# Example of loading a saved reasoning graph
loaded_graph = ReasoningGraph.load("/Users/adityashukzy/Documents/GitHub/MAT1510-Project/experiments/experiment_20251027_155017/problem_258/rollout_0")

# Verify the loaded data
print("Metrics length:", len(loaded_graph.metrics))
print("Probability distributions shape:", loaded_graph.prob_distributions[0].shape)
print("Node cutoff value:", loaded_graph.node_cutoff)

# You can now use this loaded graph for visualization or analysis

Metrics length: 185
Probability distributions shape: torch.Size([151936])
Node cutoff value: 0.0986328125


: 